In [0]:
from pyspark.sql.functions import col, sum

gold_df = spark.table("uci_retail.silver.online_retail")


In [0]:
vendas_por_pais = (
    gold_df
    .filter(
        (col("TransactionType") == "Sale") &
        (col("DataQualityStatus") == "Valid")
    )
    .groupBy("Country")
    .agg(
        sum("Revenue").alias("TotalRevenue")
    )
    .orderBy(col("TotalRevenue").desc())
)

In [0]:
display(vendas_por_pais)

Country,TotalRevenue
United Kingdom,9025222.084002146
Netherlands,285446.34
EIRE,283453.9599999986
Germany,228867.13999999844
France,209715.10999999978
Australia,138521.30999999956
Spain,61577.10999999997
Switzerland,57089.900000000096
Belgium,41196.33999999997
Sweden,38378.33000000003


In [0]:
from pyspark.sql.functions import col, sum, round

vendas_por_pais = (
    gold_df
    .filter(
        (col("TransactionType") == "Sale") &
        (col("DataQualityStatus") == "Valid")
    )
    .groupBy("Country")
    .agg(
        round(sum("Revenue"), 2).alias("TotalRevenue")
    )
    .orderBy(col("TotalRevenue").desc())
)

In [0]:
display(vendas_por_pais)

Country,TotalRevenue
United Kingdom,9025222.08
Netherlands,285446.34
EIRE,283453.96
Germany,228867.14
France,209715.11
Australia,138521.31
Spain,61577.11
Switzerland,57089.9
Belgium,41196.34
Sweden,38378.33


In [0]:
vendas_por_pais.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("uci_retail.gold.vendas_por_pais")

In [0]:
display(
    spark.table("uci_retail.gold.vendas_por_pais")
)

Country,TotalRevenue
United Kingdom,9025222.08
Netherlands,285446.34
EIRE,283453.96
Germany,228867.14
France,209715.11
Australia,138521.31
Spain,61577.11
Switzerland,57089.9
Belgium,41196.34
Sweden,38378.33


In [0]:
vendas_por_produto = (
    gold_df
    .filter(
        (col("TransactionType") == "Sale") &
        (col("DataQualityStatus") == "Valid")
    )
    .groupBy("StockCode", "Description")
    .agg(
        sum("Quantity").alias("TotalQuantity"),
        round(sum("Revenue"), 2).alias("TotalRevenue")
    )
    .orderBy(col("TotalRevenue").desc())
)

In [0]:
display(vendas_por_produto.limit(20))

StockCode,Description,TotalQuantity,TotalRevenue
DOT,DOTCOM POSTAGE,706,206248.77
22423,REGENCY CAKESTAND 3 TIER,13879,174484.74
23843,"PAPER CRAFT , LITTLE BIRDIE",80995,168469.6
85123A,WHITE HANGING HEART T-LIGHT HOLDER,37599,104340.29
47566,PARTY BUNTING,18295,99504.33
85099B,JUMBO BAG RED RETROSPOT,48474,94340.05
23166,MEDIUM CERAMIC TOP STORAGE JAR,78033,81700.92
M,Manual,7224,78110.27
POST,POSTAGE,3150,78101.88
23084,RABBIT NIGHT LIGHT,30788,66964.99


In [0]:
display(
    vendas_por_produto
    .select(
        "StockCode",
        "Description",
        "TotalQuantity",
        "TotalRevenue"
    )
    .limit(20)
)

StockCode,Description,TotalQuantity,TotalRevenue
DOT,DOTCOM POSTAGE,706,206248.77
22423,REGENCY CAKESTAND 3 TIER,13879,174484.74
23843,"PAPER CRAFT , LITTLE BIRDIE",80995,168469.6
85123A,WHITE HANGING HEART T-LIGHT HOLDER,37599,104340.29
47566,PARTY BUNTING,18295,99504.33
85099B,JUMBO BAG RED RETROSPOT,48474,94340.05
23166,MEDIUM CERAMIC TOP STORAGE JAR,78033,81700.92
M,Manual,7224,78110.27
POST,POSTAGE,3150,78101.88
23084,RABBIT NIGHT LIGHT,30788,66964.99


In [0]:
vendas_por_produto.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("uci_retail.gold.vendas_por_item")

In [0]:
display(
    spark.table("uci_retail.gold.vendas_por_item")
)

StockCode,Description,TotalQuantity,TotalRevenue
DOT,DOTCOM POSTAGE,706,206248.77
22423,REGENCY CAKESTAND 3 TIER,13879,174484.74
23843,"PAPER CRAFT , LITTLE BIRDIE",80995,168469.6
85123A,WHITE HANGING HEART T-LIGHT HOLDER,37599,104340.29
47566,PARTY BUNTING,18295,99504.33
85099B,JUMBO BAG RED RETROSPOT,48474,94340.05
23166,MEDIUM CERAMIC TOP STORAGE JAR,78033,81700.92
M,Manual,7224,78110.27
POST,POSTAGE,3150,78101.88
23084,RABBIT NIGHT LIGHT,30788,66964.99


In [0]:
from pyspark.sql.functions import date_format

vendas_por_periodo = (
    gold_df
    .filter(
        (col("TransactionType") == "Sale") &
        (col("DataQualityStatus") == "Valid")
    )
    .withColumn(
        "YearMonth",
        date_format(col("InvoiceDate"), "yyyy-MM")
    )
    .groupBy("YearMonth")
    .agg(
        sum("Quantity").alias("TotalQuantity"),
        round(sum("Revenue"), 2).alias("TotalRevenue")
    )
    .orderBy("YearMonth")
)

In [0]:
display(vendas_por_periodo)

YearMonth,TotalQuantity,TotalRevenue
2010-12,359239,823746.14
2011-01,387785,691364.56
2011-02,283555,523631.89
2011-03,377526,717639.36
2011-04,308815,537808.62
2011-05,395738,770536.02
2011-06,389213,761739.9
2011-07,401759,719221.19
2011-08,421770,759138.38
2011-09,570820,1058590.17


In [0]:
vendas_por_periodo.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("uci_retail.gold.vendas_por_periodo")

In [0]:
display(
    spark.table("uci_retail.gold.vendas_por_periodo")
)

YearMonth,TotalQuantity,TotalRevenue
2010-12,359239,823746.14
2011-01,387785,691364.56
2011-02,283555,523631.89
2011-03,377526,717639.36
2011-04,308815,537808.62
2011-05,395738,770536.02
2011-06,389213,761739.9
2011-07,401759,719221.19
2011-08,421770,759138.38
2011-09,570820,1058590.17


In [0]:
from pyspark.sql.functions import countDistinct

vendas_por_cliente = (
    gold_df
    .filter(
        (col("TransactionType") == "Sale") &
        (col("DataQualityStatus") == "Valid") &
        col("CustomerID").isNotNull()
    )
    .groupBy("CustomerID")
    .agg(
        countDistinct("InvoiceNo").alias("TotalOrders"),
        sum("Quantity").alias("TotalQuantity"),
        round(sum("Revenue"), 2).alias("TotalRevenue")
    )
    .orderBy(col("TotalRevenue").desc())
)

In [0]:
display(vendas_por_cliente.limit(20))

CustomerID,TotalOrders,TotalQuantity,TotalRevenue
14646,73,196915,280206.02
18102,60,64124,259657.3
17450,46,69993,194550.79
16446,2,80997,168472.5
14911,201,80265,143825.06
12415,21,77374,124914.53
14156,55,57885,117379.63
17511,31,64549,91062.38
16029,63,40208,81024.84
12346,1,74215,77183.6


In [0]:
vendas_por_cliente.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("uci_retail.gold.vendas_por_cliente")

In [0]:
display(
    spark.table("uci_retail.gold.vendas_por_cliente")
)

CustomerID,TotalOrders,TotalQuantity,TotalRevenue
14646,73,196915,280206.02
18102,60,64124,259657.3
17450,46,69993,194550.79
16446,2,80997,168472.5
14911,201,80265,143825.06
12415,21,77374,124914.53
14156,55,57885,117379.63
17511,31,64549,91062.38
16029,63,40208,81024.84
12346,1,74215,77183.6
